# Interpolazione polinomiale e spline


Le celle contrassegnate **DA COMPLETARE** contengono gli spazi in cui inserire il codice. 
Il codice già presente inizializza i parametri e offre funzioni utilizzabili per i grafici, che sfruttano `matplotlib`.
I valori `None` segnalano parti ancora da svolgere; i controlli nelle celle dei grafici
permettono di eseguire il modello iniziale senza errori anche prima del completamento.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import BarycentricInterpolator, CubicSpline

## E1 — Molla nonlineare

Consideriamo una molla nonlineare, per cui la forza elastica dipende dall'allungamento secondo la legge
$$F(x)=kx+\alpha x^3,\qquad k>0,\quad\alpha>0.$$
Calcolare $F$ dato $x$ è immediato. Vogliamo sfruttare questa facilità per costruire
un'approssimazione dell'operazione inversa: dato un valore di forza, stimare rapidamente l'allungamento.

Scegliamo gli allungamenti $x_i\in[0,x_{\max}]$ e calcoliamo direttamente
$$F_i=kx_i+\alpha x_i^3.$$
Le coppie da passare all'interpolazione dell'inversa sono **$(F_i,x_i)$**:
cerchiamo un polinomio $p$ tale che $p(F_i)=x_i$. Nell'interpolante, quindi,
**le forze sono le ascisse e gli allungamenti sono le ordinate**.
L'intervallo delle forze coperto dai campioni è $[0,F_{\max}]$, con
$$F_{\max}=F(x_{\max})=kx_{\max}+\alpha x_{\max}^3.$$
Confrontiamo $p(F)$ con il modello lineare $x_{\rm lin}(F)=F/k$.


### Step per la soluzione

1. **Campionamento diretto.** Scegliete $k>0$, $\alpha>0$ e $x_{\max}>0$. Spiegate perché $F(x)$ è strettamente crescente e ammette un'inversa. Scegliete 5, 9 e 17 allungamenti equidistanti in $[0,x_{\max}]$ e calcolate le corrispondenti forze con la formula diretta. Non occorre risolvere equazioni per generare i campioni.
2. **Interpolazione e verifica.** Costruite il polinomio interpolante usando la matrice di Vandermonde. Per controllare l'errore, scegliete una griglia di allungamenti `x_controllo`, più fitta e distinta dai nodi di costruzione, e calcolate `F_controllo = F(x_controllo)`. Confrontate `p(F_controllo)` con `x_controllo`, e anche `F_controllo/k` con `x_controllo`. Disegnate modello diretto, inversa interpolata ed errori. Il riferimento dell'inversa è così ottenuto in forma parametrica, senza risolvere un'equazione nonlineare. Potete controllare anche le funzioni per l'interpolazione già presenti nella libreria `scipy`, e che sono importate sopra.
3. **Parametri.** Provate, ad esempio, $k=10$, $x_{\max}=1$ e $\alpha=1,5,10$. Costruite una tabella degli errori massimi e discutete quando la linearizzazione è attendibile. A $x_{\max}$ fissato, $F_{\max}$ cambia con $\alpha$: riportatelo nella tabella. Non usate un errore relativo punto per punto nell'origine.

Il massimo sulla griglia di controllo stima l'errore sull'intervallo: non è una maggiorazione rigorosa. Usate l'interpolante per forze comprese fra quelle campionate.


In [ ]:
# Parametri di partenza. La forza massima si ricaverà dal modello nonlineare introdotto sopra.
k_molla, alpha, x_max = 10.0, 1.0, 1.0

# Esempio con un certo numero di punti -- non tutti questi serviranno per l'interpolazione!
x_controllo = np.linspace(0, x_max, 4001)

# DA COMPLETARE punti 1 e 2: prima x_nodi, poi F_nodi, poi l'inversa interpolata.
forza = None              # funzione diretta: forza(x, k, alpha)
x_nodi = None             # allungamenti scelti in [0, x_max]
F_nodi = None             # forze ottenute valutando il modello negli x_nodi
F_max = None              # forza corrispondente a x_max
interpolante = None       # oggetto p(F), costruito con (F_nodi, x_nodi)

# Anche la verifica usa soltanto valutazioni del modello diretto.
F_plot = None             # forze calcolate sulla griglia x_controllo
x_riferimento = x_controllo  # allungamenti esatti associati alle F_plot
x_lineare = None          # approssimazione lineare valutata alle stesse F_plot


In [ ]:
# Plot riutilizzabile: dopo il completamento della cella precedente.
if (forza is not None and F_plot is not None and F_nodi is not None
        and x_nodi is not None and x_lineare is not None and interpolante is not None):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(x_controllo, F_plot, label='Modello diretto')
    axes[0].plot(x_nodi, F_nodi, 'ko', ms=4, label='Campioni (x_i, F_i)')
    axes[0].set(xlabel='x [m]', ylabel='F [N]', title='1. Campionamento diretto')
    axes[1].plot(F_plot, x_riferimento, 'k--', label='Riferimento parametrico')
    axes[1].plot(F_plot, interpolante(F_plot), label='Inversa interpolata')
    axes[1].plot(F_plot, x_lineare, ':', label='Modello lineare')
    axes[1].plot(F_nodi, x_nodi, 'ko', ms=4, label='Campioni scambiati (F_i, x_i)')
    axes[1].set(xlabel='F [N]', ylabel='x [m]', title='2. Interpolazione inversa')
    axes[2].plot(F_plot, interpolante(F_plot)-x_riferimento, label='Errore interpolante')
    axes[2].plot(F_plot, x_lineare-x_riferimento, label='Errore modello lineare')
    axes[2].set(xlabel='F [N]', ylabel='errore [m]', title='3. Controllo senza inversione')
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print('Completare il campionamento diretto e l’interpolazione delle coppie (F_i, x_i).')


In [ ]:
# DA COMPLETARE punto 3: provare diversi valori di \alpha, numero e distribuzione dei nodi nel vettore x.
# Per ogni caso: scegliere x_nodi, calcolare F_nodi con la formula diretta,
# costruire p(F), valutarlo alle forze della griglia di controllo e misurare l'errore.
# Riportare anche F_max: cambia con alpha se x_max resta fissato.

## E2 — Una catena di molle come modello di una corda pesante

Vogliamo determinare la configurazione di equilibrio di una catena sotto il proprio peso e ricostruirne la forma fra i nodi. Confronteremo un **unico polinomio interpolante** con **polinomi a tratti**, costruendo con l'approccio noto come **spline**:

### Il modello fisico e le ipotesi

Consideriamo $n+2$ nodi $\mathbf r_i=(x_i,y_i)$, con $i=0,\ldots,n+1$, collegati da $n+1$ molle identiche di costante $k>0$. I **due estremi della catena** sono fissati in $\mathbf r_0=(0,y_A)$ e $\mathbf r_{n+1}=(L,y_B)$; i nodi interni portano masse $m_i>0$. Le molle sono prive di massa e il peso è diretto verso il basso; l'asse $y$ è positivo verso l'alto. I nodi che le collegano sono dotate di una massa $m_i$ assegnata e non uniforme (scegliete voi cosa utilizzare).

Assumiamo molle ideali con **lunghezza di riposo nulla**:
$$\mathbf F_{i\leftarrow j}=k(\mathbf r_j-\mathbf r_i).$$
Il modello scelto è una discretizzazione semplificata di una corda tesa e pesante.


In [ ]:
# Questa funzione di comodo si può utilizzare per visualizzare la catena calcolata.
def grafico_curve(z, curve, nodi=None, titolo='', ylabel='y'):
    """curve: dizionario {etichetta: valori}; nodi: coppia (ascisse, ordinate)."""
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for nome, valori in curve.items():
        ax.plot(z, valori, label=nome)
    if nodi is not None:
        ax.plot(*nodi, 'ko', ms=4, label='Nodi')
    ax.set(xlabel='x', ylabel=ylabel, title=titolo)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()


### E2.1 — Determinare l'equilibrio

1. Scrivete la risultante delle forze su un nodo interno e imponete che sia nulla.
2. Separate le equazioni per $x_i$ e $y_i$. Costruite la matrice e i termini noti del sistema per i soli nodi interni. Scrivete esplicitamente il caso $n=3$.
3. Spiegate come entrano i vincoli ai capi e perché l'equilibrio è unico. Attenzione al caso $n=1$: entrambi i vincoli contribuiscono alla stessa equazione.
4. Dimostrate che le ascisse dei nodi sono equidistanti. Implementate `equilibrio` usando `np.linalg.solve`, poi controllate il residuo e disegnate la catena.

**Dati per gli esperimenti.** Ponete $h=L/(n+1)$, $k=H/h$ e
$m_i=m$
In ciascuna catena tutte le molle hanno la stessa costante $k$, e i nodi di collegamento hanno peso $m$.


In [ ]:
L, g, m, beta, H = 1.0, 9.81, 1.0, 0.8, 20.0
y_A, y_B = 0.0, 0.0
n = 10
h = L / (n + 1)
k = H / h

def equilibrio(n, L, k, rho0, beta, g, y_A=0.0, y_B=0.0):
    # DA COMPLETARE E2.1: masse, matrice, termini noti, due sistemi.
    # Restituire (x, y, A, b_y), includendo i capi negli array x e y.
    return None

risultato = equilibrio(n, L, k, m, beta, g, y_A, y_B)
x, y = None, None
if risultato is not None:
    x, y, A, b_y = risultato
    grafico_curve(x, {'Catena': y}, nodi=(x, y),
                  titolo='E2 — Configurazione di equilibrio', ylabel='y [m]')
else:
    print('Completare equilibrio per visualizzare la catena.')


### E2.2 — Interpolazione globale

Poniamo $h=L/(n+1)$ e assegniamo le masse mediante
$$m_i=\rho(x_i)h,\qquad \rho(x)=\rho_0[1+\beta\cos(2\pi x/L)],\quad |\beta|<1.$$

La componente orizzontale della forza di ogni molla è $kh$. 

Dividendo l'equazione verticale per $h$, riconosciamo una differenza seconda:
$$H\frac{y_{i-1}-2y_i+y_{i+1}}{h^2}=\rho(x_i)g.$$


**Un polinomio su tutti i nodi.**

Con $n+2$ nodi distinti esiste un unico polinomio $p$ di grado al più $n+1$ tale che $p(x_i)=y_i$. 

Il polinomio usa tutti i dati su tutto l'intervallo. All'aumentare del grado possono comparire oscillazioni e sensibilità numerica, soprattutto con nodi equidistanti: il miglioramento non è garantito. In questo esempio regolare può anche dare ottimi risultati. Non spostiamo i nodi fisici sui nodi di Chebyshev: cambieremmo la catena del modello.


**Da svolgere E2.2.** Implementate la formula di $y_*$ data sopra e la sua derivata.
Costruite il polinomio globale sui nodi di equilibrio. Confrontate forma ed errore su una griglia fitta e distinguete l'errore ai nodi da quello fra i nodi.


In [ ]:
xx = np.linspace(0, L, 2001)
corda_riferimento = None       # DA COMPLETARE: funzione di z
pendenza_riferimento = None   # DA COMPLETARE: funzione di z
polinomio = None              # DA COMPLETARE: interpolante sui nodi x, y

if corda_riferimento is not None and polinomio is not None and x is not None:
    grafico_curve(xx, {'Corda continua': corda_riferimento(xx),
                      'Polinomio globale': polinomio(xx)},
                  nodi=(x, y), titolo='E2 — Interpolazione globale', ylabel='y [m]')


### E2.3 — Dall'interpolazione a tratti alle spline

Invece di usare un solo polinomio di grado elevato, scegliamo un polinomio $s_i$ di grado basso su ciascun intervallo $[x_i,x_{i+1}]$. Una **spline polinomiale** è una funzione così costruita, con condizioni di raccordo nei nodi interni.

Occorre distinguere i **raccordi fra tratti** dalle **condizioni ai due estremi esterni** $x_0$ e $x_{n+1}$.

| Regolarità nei nodi interni | Condizioni fra due tratti adiacenti | Aspetto geometrico |
| :--- | :--- | :--- |
| $C^0$ | valori uguali | nessun salto, ma possibili spigoli |
| $C^1$ | valori e derivate prime uguali | tangente continua |
| $C^2$ | valori, derivate prime e seconde uguali | anche la derivata seconda è continua |

Se abbiamo $N=n+1$ intervalli, l'interpolazione ai due capi di ogni tratto dà $2N$ condizioni. Una retta per intervallo ha $2N$ coefficienti: non resta libertà per imporre altre condizioni. Una quadratica ha $3N$ coefficienti: dopo l'interpolazione e i $N-1$ raccordi $C^1$, resta **una condizione**. Una cubica ha $4N$ coefficienti: dopo l'interpolazione e i $2(N-1)$ raccordi $C^1,C^2$, restano **due condizioni**.

#### Interpolazione lineare a tratti: continuità $C^0$

Su $[x_i,x_{i+1}]$, posto $h_i=x_{i+1}-x_i$, si usa
$$s_i(x)=y_i+\frac{y_{i+1}-y_i}{h_i}(x-x_i).$$
Il grafico coincide con la spezzata che unisce i nodi: è già una spline di grado 1. Le pendenze dei due tratti adiacenti possono essere diverse.


**Da svolgere E2.3.** Implementate la spezzata con `np.interp` e calcolate le pendenze dei tratti. Misurate i loro salti nei nodi interni.


In [ ]:
lineare_a_tratti = None  # DA COMPLETARE: funzione che valuta la spezzata su z
pendenze_secanti = None  # DA COMPLETARE: array, uno per intervallo


### E2.4 — Spline quadratica $C^1$

Nel tratto $[x_i,x_{i+1}]$ poniamo
$$s_i(x)=y_i+d_i(x-x_i)+a_i(x-x_i)^2.$$
Qui $d_i$ rappresenta la derivata al capo sinistro. Per raccordare le tangenti deve coincidere con la derivata finale del tratto precedente.

**Da svolgere.** Esprimete $a_i$ in termini di $d_i$ e della secante $\delta_i$ imponendo l'interpolazione a destra. Ricavate poi una ricorrenza per $d_{i+1}$ imponendo $C^1$. Scegliete inizialmente $d_0=\delta_0$; provate poi $d_0=y_*'(0)$. Quale condizione impone la prima scelta sul primo tratto? Implementate la costruzione e calcolate i salti della derivata seconda.


In [ ]:
def costruisci_quadratica(xn, yn, d0=None):
    # DA COMPLETARE E2.4: coefficienti e funzione di valutazione.
    # Restituire (valuta, d, a), con d di lunghezza len(xn), a di lunghezza len(xn)-1.
    return None

quadratica = None
# Dopo il completamento: quadratica, d_quad, a_quad = costruisci_quadratica(x, y)


### E2.5 — Spline cubica: continuità $C^2$

Un modo per costruirla è cercare le derivate seconde $M_i=s''(x_i)$. Nel tratto $[x_i,x_{i+1}]$ il polinomio cubico si può scrivere come
$$s_i(x)=\frac{M_i(x_{i+1}-x)^3}{6h_i}
+\frac{M_{i+1}(x-x_i)^3}{6h_i}
+\left(y_i-\frac{M_i h_i^2}{6}\right)\frac{x_{i+1}-x}{h_i}
+\left(y_{i+1}-\frac{M_{i+1}h_i^2}{6}\right)\frac{x-x_i}{h_i}.$$
Questa forma interpola i valori e raccorda le derivate seconde. Imponendo anche la continuità delle derivate prime si ricava, nei nodi interni,
$$h_{i-1}M_{i-1}+2(h_{i-1}+h_i)M_i+h_iM_{i+1}
=6(\delta_i-\delta_{i-1}).$$
È un altro **sistema lineare tridiagonale**. Mancano le due condizioni esterne:

| Scelta | Condizioni agli estremi esterni | Significato |
| :--- | :--- | :--- |
| Naturale | $M_0=M_N=0$ | derivata seconda nulla ai capi |
| Vincolata (*clamped*) | $s'(x_0)=d_A$, $s'(x_N)=d_B$ | pendenze assegnate |
| *Not-a-knot* | terza derivata continua in $x_1$ e $x_{N-1}$ | i primi due tratti e gli ultimi due sono ciascuno un unico cubico |

I vincoli fisici fissano soltanto **le posizioni** dei capi: non impongono né pendenza nulla né derivata seconda nulla. La spline naturale è una scelta di ricostruzione; non è automaticamente quella più fedele alla corda.

Per la spline vincolata le due righe esterne sono
$$2h_0M_0+h_0M_1=6(\delta_0-d_A),\qquad
h_{N-1}M_{N-1}+2h_{N-1}M_N=6(d_B-\delta_{N-1}).$$
Nel confronto useremo le pendenze della soluzione continua per mostrare l'effetto di informazioni aggiuntive. In un problema reale andrebbero misurate, stimate o assegnate dal modello. In SciPy `bc_type='clamped'` impone specificamente **pendenze nulle**: per valori diversi si usa `((1, d_A), (1, d_B))`.

**Da svolgere.** Derivate le equazioni per $M_i$ uguagliando $s'_{i-1}(x_i)$ e $s'_i(x_i)$. Poi completate a mano il sistema della spline naturale per quattro nodi.


**Implementazione E2.5.** Costruite il sistema per le derivate seconde della spline naturale e implementate la formula di valutazione a tratti, senza chiamare routine di interpolazione al suo interno.
Confrontate poi il risultato con `CubicSpline`. Costruite anche le cubiche vincolata e not-a-knot e confrontatele, prestando attenzione ai capi.


In [ ]:
def costruisci_cubica_naturale(xn, yn):
    # DA COMPLETARE: sistema per M e valutazione a tratti.
    # Restituire (valuta, M).
    return None

cubica_manual = None
cubica_naturale = None
cubica_vincolata = None
cubica_notaknot = None
